# Perseptron v2 - 05 MAP@12 Ranking Evaluation

Bu notebook proposal'daki ana metrik olan **MAP@12** degerlendirmesini calistirir. Tabular-only, image-history ve late-fusion checkpointleri ayni candidate havuzu uzerinde karsilastirilir.

Ayrica `late_fusion_hybrid` sonradan eklenen post-ranking/reranking deneyi olarak ayri satirlarda raporlanir; ana model olarak yorumlanmaz.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

def find_source_project():
    candidates = [Path('/kaggle/working/perseptron_project'), Path.cwd(), Path('/kaggle/input/perseptron-project')]
    candidates += [path for path in Path('/kaggle/input').glob('*') if path.is_dir()]
    for path in candidates:
        if (path / 'src' / 'proposal_v2').exists():
            return path
    return Path.cwd()

SOURCE_PROJECT_DIR = find_source_project()
WORK_PROJECT_DIR = Path('/kaggle/working/perseptron_project_work') if Path('/kaggle').exists() else SOURCE_PROJECT_DIR
if SOURCE_PROJECT_DIR != WORK_PROJECT_DIR:
    shutil.copytree(SOURCE_PROJECT_DIR, WORK_PROJECT_DIR, dirs_exist_ok=True)

PROJECT_DIR = WORK_PROJECT_DIR
os.environ['PERSEPTRON_PROJECT_DIR'] = str(PROJECT_DIR)
os.environ['PYTHONPATH'] = str(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('SOURCE_PROJECT_DIR =', SOURCE_PROJECT_DIR)
print('PROJECT_DIR =', PROJECT_DIR)

def restore_previous_outputs():
    if not Path('/kaggle/input').exists():
        return
    for input_root in Path('/kaggle/input').glob('*'):
        if input_root == SOURCE_PROJECT_DIR:
            continue
        for relative in ['reports/proposal_v2', 'models/proposal_v2']:
            source = input_root / relative
            target = PROJECT_DIR / relative
            if source.exists():
                target.mkdir(parents=True, exist_ok=True)
                shutil.copytree(source, target, dirs_exist_ok=True)
                print('Restored', source, '->', target)

restore_previous_outputs()

def run_module(module, *args):
    command = [sys.executable, '-m', module, *map(str, args)]
    print('RUN:', ' '.join(command))
    subprocess.run(command, cwd=PROJECT_DIR, check=True)


## Parametreler

Full kosuda `SAMPLE_CUSTOMERS = None` yap. Smoke icin 100 gibi kucuk bir sayi kalabilir.

In [ ]:
FAST_RUN = True
FOLD_ID = 0
SAMPLE_CUSTOMERS = 100 if FAST_RUN else None
CANDIDATE_LIMIT = 5000
VISUAL_NEIGHBORS = 3000
CO_PURCHASE_PER_ITEM = 300
HYBRID_WEIGHTS = '0.25,0.45,0.65'


## Ranking Evaluation

Bu hucre musteri bazli prediction listeleri uretir ve MAP@12 / Precision@10 / Recall@10 metriklerini kaydeder.

In [ ]:
args = [
    '--fold-id', FOLD_ID,
    '--candidate-limit', CANDIDATE_LIMIT,
    '--visual-neighbors', VISUAL_NEIGHBORS,
    '--co-purchase-per-item', CO_PURCHASE_PER_ITEM,
    '--hybrid-weights', HYBRID_WEIGHTS,
]
if SAMPLE_CUSTOMERS is not None:
    args += ['--sample-customers', SAMPLE_CUSTOMERS]
run_module('src.proposal_v2.ranking', *args)
run_module('src.proposal_v2.aggregate_results')
